# Sprint 1: Baseline RAG on PubMedQA

## 1. Title and research objective

Sprint 1 establishes the relevance-only RAG baseline for the broader *Context Matters* study. It provides the focused-evidence PubMedQA control needed before later sprints study diversity-aware retrieval on multi-hop and ambiguous information needs.

**RQ1:** How does relevance-only `WITH_CONTEXT` generation compare with `WITHOUT_CONTEXT` generation on PubMedQA?

This notebook keeps two evidence layers separate:

- **Historical Sprint 1 (`HISTORICAL_OBSERVED`)**: verified relevance-only outputs from the original project run.
- **Current controlled replication**: the frozen three-LLM design, whose generation and evaluation results are not yet available.

Historical observations are never relabelled or combined with current-protocol outputs as though they came from one controlled experiment.

## 2. Experimental design

The controlled Sprint-1 baseline uses the complete expert-labelled PubMedQA PQA-L1000 population (1,000 questions) and the project corpus of 3,358 context-section documents. All retrievers operate over the same logical corpus.

- Retrievers: BM25, DPR, Contriever, and ColBERTv2.
- Shared first-stage candidate pool: Top-20.
- Final relevance-only context: the first five ranked passages.
- Current generators: `llama-3.3-70b`, `gemma4-26b`, and `ministral-3-14b`.
- Conditions: `WITHOUT_CONTEXT` and relevance-only `WITH_CONTEXT`.
- Within each controlled comparison, prompt identity and decoding remain fixed; only context presence or selected context changes.

In [ ]:
from __future__ import annotations

import hashlib
import json
import os
import sys
import tempfile
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import Markdown, display

def find_repository_root(start: Path | None = None) -> Path:
    current = (start or Path.cwd()).resolve()
    for candidate in (current, *current.parents):
        if (candidate / "AGENTS.md").is_file() and (candidate / "src").is_dir():
            return candidate
    raise RuntimeError("Repository root not found; run this notebook from the project tree.")

REPO_ROOT = find_repository_root()
for import_root in (REPO_ROOT, REPO_ROOT / "src"):
    if str(import_root) not in sys.path:
        sys.path.insert(0, str(import_root))

HISTORICAL_RAW_DIR = REPO_ROOT / "results/sprint1/raw"
SAMPLE_MANIFEST_PATH = REPO_ROOT / "artifacts/sample_manifests/pubmedqa_sample_manifest_v2.json"
CORPUS_MANIFEST_PATH = REPO_ROOT / "artifacts/corpus_manifests/pubmedqa_corpus_manifest_v1.json"
HISTORICAL_INVENTORY_PATH = REPO_ROOT / "artifacts/audit_inventories/sprint1_historical_artifact_inventory_v1.json"
RUN_REGISTRY_PATH = REPO_ROOT / "artifacts/run_registry/run_registry_v1.jsonl"

RETRIEVERS = ("bm25", "dpr", "contriever", "colbertv2")
RETRIEVER_LABELS = {
    "bm25": "BM25",
    "dpr": "DPR",
    "contriever": "Contriever",
    "colbertv2": "ColBERTv2",
}
CURRENT_LLMS = ("llama-3.3-70b", "gemma4-26b", "ministral-3-14b")
CANDIDATE_POOL = 20
FINAL_TOP_K = 5
EXPECTED_SAMPLE_COUNT = 1_000
EXPECTED_CORPUS_COUNT = 3_358

print(f"Repository: {REPO_ROOT}")
print("Safety mode: artifact loading only; no retriever, generator, evaluator, or downloader is called.")

In [ ]:
design_table = pd.DataFrame(
    [
        ("Dataset", "PubMedQA PQA-L1000"),
        ("Questions", f"{EXPECTED_SAMPLE_COUNT:,}"),
        ("Corpus documents", f"{EXPECTED_CORPUS_COUNT:,}"),
        ("Retrievers", ", ".join(RETRIEVER_LABELS[r] for r in RETRIEVERS)),
        ("Candidate pool", str(CANDIDATE_POOL)),
        ("Final context", f"Top-{FINAL_TOP_K} relevance-only passages"),
        ("Current LLMs", ", ".join(CURRENT_LLMS)),
        ("Context modes", "WITHOUT_CONTEXT; WITH_CONTEXT relevance-only"),
    ],
    columns=["Design element", "Frozen value"],
)
display(design_table.style.hide(axis="index"))

## 3. Historical Sprint 1 provenance

The historical layer is loaded only from verified repository artifacts and is labelled `HISTORICAL_OBSERVED`. Its generator was `ministral-3-14b`, with temperature 0, a 512-token maximum, and the historical context-only prompt. The current generation protocol instead uses a byte-frozen system/user prompt contract and a 256-token PubMedQA maximum.

Consequently, historical Ministral outputs must not be mixed with future Llama or Gemma outputs—or with a current-protocol Ministral rerun—as if they formed one controlled experiment. The known historical BM25 one-document anomaly is retained rather than repaired.

In [ ]:
from scripts.validate_sprint1_historical_artifacts import (
    RESULT_FILENAMES_BY_RETRIEVER,
    validate_historical_sprint1_artifacts,
)

if not HISTORICAL_INVENTORY_PATH.is_file():
    raise FileNotFoundError(f"Required audit inventory is missing: {HISTORICAL_INVENTORY_PATH}")
historical_inventory = json.loads(HISTORICAL_INVENTORY_PATH.read_text(encoding="utf-8"))

# The existing project validator writes only a temporary audit copy. It does not
# modify the frozen raw results or the tracked audit inventory.
with tempfile.TemporaryDirectory(prefix="sprint1-notebook-audit-") as temp_dir:
    historical_validation = validate_historical_sprint1_artifacts(
        input_dir=HISTORICAL_RAW_DIR,
        output_path=Path(temp_dir) / "validation.json",
        repository_root=REPO_ROOT,
    )

historical_frames = {
    retriever: pd.read_csv(HISTORICAL_RAW_DIR / filename)
    for retriever, filename in RESULT_FILENAMES_BY_RETRIEVER.items()
}
historical_audit_table = pd.DataFrame(
    [
        ("Evidence role", historical_validation["historical_contract"]["evidence_role"]),
        ("Validated rows", historical_validation["aggregate"]["total_rows"]),
        ("Status counts", historical_validation["aggregate"]["status_counts"]),
        ("Known preserved anomalies", historical_validation["aggregate"]["known_anomaly_count"]),
        ("Generator", historical_validation["historical_contract"]["generator"]),
        ("Historical max tokens", historical_validation["historical_contract"]["max_tokens"]),
    ],
    columns=["Audit field", "Verified value"],
)
display(historical_audit_table.style.hide(axis="index"))

## 4. Dataset and corpus validation

The frozen sample and corpus manifests provide the canonical identities. The optional text preview below reads only the already-cached immutable PubMedQA Arrow file at the frozen revision. It never downloads or reconstructs a dataset. If that cache is absent, manifest validation still runs and the text preview reports an explicit unavailable status.

In [ ]:
from scripts.build_corpus_manifests import (
    PUBMEDQA_REVISION,
    load_frozen_pubmedqa_corpus_manifest,
    load_frozen_pubmedqa_sample_manifest,
)

sample_manifest = load_frozen_pubmedqa_sample_manifest(SAMPLE_MANIFEST_PATH)
corpus_manifest = load_frozen_pubmedqa_corpus_manifest(CORPUS_MANIFEST_PATH)
if sample_manifest.actual_sample_size != EXPECTED_SAMPLE_COUNT:
    raise ValueError("Frozen PubMedQA sample count is not 1,000.")
if corpus_manifest.document_count != EXPECTED_CORPUS_COUNT:
    raise ValueError("Frozen PubMedQA corpus count is not 3,358.")

manifest_table = pd.DataFrame(
    [
        ("Sample count", sample_manifest.actual_sample_size),
        ("Sample manifest", sample_manifest.manifest_id),
        ("Corpus document count", corpus_manifest.document_count),
        ("Corpus manifest", corpus_manifest.corpus_manifest_id),
        ("Pinned source revision", PUBMEDQA_REVISION),
        ("Corpus construction", corpus_manifest.construction_algorithm),
    ],
    columns=["Validation field", "Value"],
)
display(manifest_table.style.hide(axis="index"))

In [ ]:
from retrieval_artifacts import document_content_sha256, query_text_sha256

def locate_pinned_pubmedqa_arrow() -> Path | None:
    configured_cache = os.environ.get("HF_DATASETS_CACHE")
    cache_roots = []
    if configured_cache:
        cache_roots.append(Path(configured_cache).expanduser())
    cache_roots.append(Path.home() / ".cache/huggingface/datasets")
    relative = Path(
        "qiaojin___pub_med_qa/pqa_labeled/0.0.0"
    ) / PUBMEDQA_REVISION / "pub_med_qa-train.arrow"
    matches = [root / relative for root in cache_roots if (root / relative).is_file()]
    unique = list(dict.fromkeys(path.resolve() for path in matches))
    if len(unique) > 1:
        raise RuntimeError(f"Multiple pinned PubMedQA Arrow files found: {unique}")
    return unique[0] if unique else None

def validate_local_source_and_examples(arrow_path: Path | None, example_count: int = 3):
    historical_questions = historical_frames["bm25"][["qa_id", "question"]].copy()
    if arrow_path is None:
        questions = historical_questions.head(example_count).rename(columns={"qa_id": "sample_id"})
        return "SOURCE TEXT UNAVAILABLE LOCALLY", questions, pd.DataFrame()

    from datasets import Dataset

    source = Dataset.from_file(str(arrow_path))  # Direct local read; no Hub call.
    if len(source) != EXPECTED_SAMPLE_COUNT:
        raise ValueError(f"Pinned local source has {len(source)} rows, expected 1,000.")

    source_rows = [source[index] for index in range(EXPECTED_SAMPLE_COUNT)]
    for entry, row in zip(sample_manifest.entries, source_rows, strict=True):
        if str(entry.source_sample_id) != str(row["pubid"]):
            raise ValueError(f"Source PubMed ID mismatch at sample {entry.position}.")
        if query_text_sha256(row["question"]) != entry.query_text_sha256:
            raise ValueError(f"Question hash mismatch at sample {entry.position}.")

    flat_sections = []
    for row in source_rows:
        for ordinal, section in enumerate(row["context"]["contexts"]):
            flat_sections.append((row["pubid"], ordinal, section.strip()))
    if len(flat_sections) != EXPECTED_CORPUS_COUNT:
        raise ValueError(f"Pinned local source has {len(flat_sections)} sections, expected 3,358.")
    for entry, (pubid, ordinal, section) in zip(corpus_manifest.entries, flat_sections, strict=True):
        expected_source_id = f"pubmedqa:pubid:{pubid}:context:{ordinal}"
        if entry.source_document_id != expected_source_id:
            raise ValueError(f"Source-document identity mismatch at corpus position {entry.position}.")
        if document_content_sha256(section) != entry.retrieval_content_sha256:
            raise ValueError(f"Document-content hash mismatch at corpus position {entry.position}.")

    questions = pd.DataFrame(
        [{"sample_id": i, "pubid": source_rows[i]["pubid"], "question": source_rows[i]["question"]} for i in range(example_count)]
    )
    documents = pd.DataFrame(
        [
            {
                "document_id": corpus_manifest.entries[i].doc_id,
                "source_document_id": corpus_manifest.entries[i].source_document_id,
                "body_preview": flat_sections[i][2][:300] + ("…" if len(flat_sections[i][2]) > 300 else ""),
            }
            for i in range(example_count)
        ]
    )
    return "PINNED LOCAL SOURCE VALIDATED", questions, documents

local_arrow_path = locate_pinned_pubmedqa_arrow()
source_status, example_questions, example_documents = validate_local_source_and_examples(local_arrow_path)
display(Markdown(f"**Local source status:** `{source_status}`"))
display(Markdown("**Example questions**"))
display(example_questions.style.hide(axis="index"))
display(Markdown("**Example corpus documents**"))
if example_documents.empty:
    display(Markdown("Document text is unavailable because the exact pinned Arrow cache is not present. No substitute was loaded."))
else:
    display(example_documents.style.hide(axis="index"))

## 5. Retriever overview

The comparison spans one sparse lexical method, two dense dual-encoder methods, and one late-interaction method. The table is generated from the repository's dependency-neutral frozen configurations rather than guessed defaults.

In [ ]:
from retrievers.bm25_config import BM25_CONFIG
from retrievers.colbert_config import COLBERT_CONFIG
from retrievers.contriever_config import CONTRIEVER_CONFIG
from retrievers.dpr_config import DPR_CONFIG

retriever_table = pd.DataFrame(
    [
        {
            "Retriever": "BM25",
            "Family": "Sparse lexical",
            "Project model/checkpoint": "rank_bm25.BM25Okapi (no neural checkpoint)",
            "Frozen representation / scoring": f"whitespace tokens; k1={BM25_CONFIG.k1}, b={BM25_CONFIG.b}; native BM25 score",
            "Index": BM25_CONFIG.index_type,
        },
        {
            "Retriever": "DPR",
            "Family": "Supervised dense dual encoder",
            "Project model/checkpoint": f"{DPR_CONFIG.question_model_id} / {DPR_CONFIG.context_model_id}",
            "Frozen representation / scoring": f"{DPR_CONFIG.representation}, {DPR_CONFIG.embedding_dimension}d, {DPR_CONFIG.score_semantics}",
            "Index": DPR_CONFIG.index_type,
        },
        {
            "Retriever": "Contriever",
            "Family": "Unsupervised dense encoder",
            "Project model/checkpoint": f"{CONTRIEVER_CONFIG.model_id}@{CONTRIEVER_CONFIG.model_revision[:12]}…",
            "Frozen representation / scoring": f"{CONTRIEVER_CONFIG.pooling}, {CONTRIEVER_CONFIG.embedding_dimension}d, {CONTRIEVER_CONFIG.score_semantics}",
            "Index": CONTRIEVER_CONFIG.index_type,
        },
        {
            "Retriever": "ColBERTv2",
            "Family": "Token-level late interaction",
            "Project model/checkpoint": f"{COLBERT_CONFIG.checkpoint_id}@{COLBERT_CONFIG.checkpoint_revision[:12]}…",
            "Frozen representation / scoring": f"{COLBERT_CONFIG.interaction}, dim={COLBERT_CONFIG.dim}, {COLBERT_CONFIG.similarity}",
            "Index": COLBERT_CONFIG.index_engine,
        },
    ]
)
display(retriever_table.style.hide(axis="index"))

## 6. Candidate retrieval validation

This audit reads existing per-query candidate artifacts through the project schema validator. It checks the frozen sample/corpus identities, exact query hashes, Top-20 cardinality, ranks, document identities, and content hashes. It does not instantiate a retriever, load a model/index, or regenerate missing files.

In [ ]:
from retrieval_artifacts import read_candidate_artifact

sample_entries_by_position = {entry.position: entry for entry in sample_manifest.entries}
corpus_entries_by_position = {entry.position: entry for entry in corpus_manifest.entries}

def validate_candidate_artifact_against_manifests(path: Path, expected_retriever: str, position: int) -> None:
    artifact = read_candidate_artifact(path)
    sample_entry = sample_entries_by_position[position]
    if artifact.sample_id != sample_entry.sample_id:
        raise ValueError("sample_id does not match frozen manifest")
    if query_text_sha256(artifact.query_text) != sample_entry.query_text_sha256:
        raise ValueError("query text does not match frozen manifest")
    if artifact.dataset.sample_manifest_id != sample_manifest.manifest_id:
        raise ValueError("sample-manifest identity mismatch")
    if artifact.dataset.sample_manifest_sha256 != sample_manifest.sha256:
        raise ValueError("sample-manifest hash mismatch")
    if artifact.corpus.corpus_id != corpus_manifest.corpus_manifest_id:
        raise ValueError("corpus-manifest identity mismatch")
    if artifact.corpus.manifest_sha256 != corpus_manifest.sha256:
        raise ValueError("corpus-manifest hash mismatch")
    if artifact.corpus.document_count != EXPECTED_CORPUS_COUNT:
        raise ValueError("corpus document count mismatch")
    if artifact.retriever.retriever_name != expected_retriever:
        raise ValueError("retriever identity mismatch")
    if artifact.requested_top_n != CANDIDATE_POOL or len(artifact.candidates) != CANDIDATE_POOL:
        raise ValueError("candidate artifact is not a complete Top-20 pool")
    for candidate in artifact.candidates:
        if candidate.corpus_position is None:
            raise ValueError("candidate has no corpus position")
        manifest_entry = corpus_entries_by_position.get(candidate.corpus_position)
        if manifest_entry is None:
            raise ValueError("candidate corpus position is outside the manifest")
        if candidate.document_id != manifest_entry.doc_id:
            raise ValueError("candidate document ID does not match its corpus position")
        if candidate.source_document_id != manifest_entry.source_document_id:
            raise ValueError("candidate source document ID mismatch")
        if candidate.document_content_sha256 != manifest_entry.retrieval_content_sha256:
            raise ValueError("candidate document-content hash mismatch")

def audit_candidate_directory(retriever: str) -> dict:
    directory = REPO_ROOT / f"artifacts/candidates/pubmedqa/{retriever}"
    expected_paths = {directory / f"sample_{position:04d}.json" for position in range(EXPECTED_SAMPLE_COUNT)}
    present_paths = set(directory.glob("*.json")) if directory.is_dir() else set()
    unexpected = sorted(p for p in present_paths if p not in expected_paths)
    invalid = []
    valid = 0
    for position in range(EXPECTED_SAMPLE_COUNT):
        path = directory / f"sample_{position:04d}.json"
        if not path.is_file():
            continue
        try:
            validate_candidate_artifact_against_manifests(path, retriever, position)
            valid += 1
        except Exception as error:
            invalid.append(f"{path.name}: {type(error).__name__}: {error}")
    available = len(present_paths & expected_paths)
    missing = EXPECTED_SAMPLE_COUNT - available
    if invalid or unexpected:
        status = "INVALID — STOP"
    elif valid == EXPECTED_SAMPLE_COUNT:
        status = "COMPLETE REUSABLE"
    elif valid:
        status = "PARTIAL REUSABLE"
    else:
        status = "UNAVAILABLE LOCALLY"
    return {
        "Retriever": RETRIEVER_LABELS[retriever],
        "Expected": EXPECTED_SAMPLE_COUNT,
        "Available": available,
        "Valid reusable": valid,
        "Missing": missing,
        "Invalid": len(invalid) + len(unexpected),
        "Candidate pool": CANDIDATE_POOL,
        "Status": status,
        "Details": invalid[:3] + [f"Unexpected file: {p.name}" for p in unexpected[:3]],
    }

In [ ]:
candidate_audits = [audit_candidate_directory(retriever) for retriever in RETRIEVERS]
candidate_table = pd.DataFrame([{key: value for key, value in row.items() if key != "Details"} for row in candidate_audits])
display(candidate_table.style.hide(axis="index"))
candidate_errors = {row["Retriever"]: row["Details"] for row in candidate_audits if row["Details"]}
if candidate_errors:
    raise RuntimeError(f"Invalid or unexpected candidate artifacts found: {candidate_errors}")
display(Markdown("Existing valid artifacts are preserved. Missing artifacts remain missing and are not generated by this notebook."))

## 7. Historical Sprint 1 results

The table below recomputes metric means from the four verified raw CSVs and cross-checks them against the stored historical summary. Recall@5 and MRR describe recovery/ranking of dataset-provided PubMedQA context sections in the constructed project corpus. Historical F1 and ROUGE-L are lexical answer metrics against `long_answer`; they are not the current protocol's categorical decision accuracy.

In [ ]:
summary_path = HISTORICAL_RAW_DIR / "fullrag_summary_top5.csv"
if not summary_path.is_file():
    raise FileNotFoundError(f"Historical summary is missing: {summary_path}")
stored_summary = pd.read_csv(summary_path).set_index("retriever")

historical_rows = []
for retriever in RETRIEVERS:
    frame = historical_frames[retriever]
    if len(frame) != EXPECTED_SAMPLE_COUNT or frame["qa_id"].nunique() != EXPECTED_SAMPLE_COUNT:
        raise ValueError(f"Historical {retriever} rows do not cover 1,000 unique questions.")
    row = {
        "Retriever": RETRIEVER_LABELS[retriever],
        "Recall@5": frame["recall_at_k"].mean(),
        "MRR": frame["mrr"].mean(),
        "F1": frame["f1"].mean(),
        "ROUGE-L": frame["rouge_l"].mean(),
    }
    cross_checks = {
        "Recall@5": stored_summary.loc[retriever, "avg_recall_at_k"],
        "MRR": stored_summary.loc[retriever, "avg_mrr"],
        "F1": stored_summary.loc[retriever, "avg_f1"],
        "ROUGE-L": stored_summary.loc[retriever, "avg_rouge_l"],
    }
    for metric, expected in cross_checks.items():
        if not np.isclose(row[metric], expected, rtol=0.0, atol=1e-12):
            raise ValueError(f"Historical {retriever} {metric} does not match stored summary.")
    historical_rows.append(row)

historical_results = pd.DataFrame(historical_rows).set_index("Retriever")
display(historical_results.style.format("{:.6f}"))

In [ ]:
try:
    import matplotlib.pyplot as plt
except ModuleNotFoundError:
    display(Markdown("**Figure unavailable:** install the project notebook environment with matplotlib; no substitute plotting library is used."))
else:
    fig, axes = plt.subplots(1, 2, figsize=(13, 4.8), constrained_layout=True)
    x = np.arange(len(historical_results.index))
    width = 0.36
    for axis, columns, title in (
        (axes[0], ("Recall@5", "MRR"), "Historical retrieval metrics"),
        (axes[1], ("F1", "ROUGE-L"), "Historical lexical generation metrics"),
    ):
        for offset, column in zip((-width / 2, width / 2), columns, strict=True):
            axis.bar(x + offset, historical_results[column], width, label=column)
        axis.set_title(title)
        axis.set_xlabel("Retriever")
        axis.set_ylabel("Mean score")
        axis.set_xticks(x, historical_results.index, rotation=20, ha="right")
        axis.set_ylim(0, 1)
        axis.grid(axis="y", alpha=0.25)
        axis.legend()
    fig.suptitle("Verified historical Sprint-1 PubMedQA observations", fontsize=14)
    plt.show()

## 8. Historical interpretation

Within the verified historical run, ColBERTv2 had the strongest stored retrieval scores and DPR the weakest. The historical lexical generation metrics varied much less across retrievers than Recall@5 and MRR. These are descriptive observations from exposed historical evidence—not causal, universal, or current-protocol conclusions. Prompt, output contract, token limit, and evaluator differences prevent treating them as direct results of the pending controlled replication.

## 9. Current controlled replication design

The present Sprint-1 replication isolates RQ1 and the four relevance-only retriever baselines. It is narrower than the later full diversification matrix. `WITHOUT_CONTEXT` is generated exactly once per question × LLM and reused across retriever comparisons.

In [ ]:
without_context_count = EXPECTED_SAMPLE_COUNT * len(CURRENT_LLMS)
with_context_count = EXPECTED_SAMPLE_COUNT * len(RETRIEVERS) * len(CURRENT_LLMS)
current_target_count = without_context_count + with_context_count
assert (without_context_count, with_context_count, current_target_count) == (3_000, 12_000, 15_000)

replication_matrix = pd.DataFrame(
    [
        {"Condition": "WITHOUT_CONTEXT", "Calculation": "1,000 questions × 3 LLMs", "Generations": without_context_count},
        {"Condition": "WITH_CONTEXT relevance-only", "Calculation": "1,000 × 4 retrievers × 3 LLMs", "Generations": with_context_count},
        {"Condition": "Total current Sprint-1 target", "Calculation": "3,000 + 12,000", "Generations": current_target_count},
    ]
)
display(replication_matrix.style.hide(axis="index").format({"Generations": "{:,}"}))
display(Markdown("`WITHOUT_CONTEXT` has no retriever key and is not multiplied fourfold."))

## 10. Current generation-results loader

Current-protocol outputs are discoverable only through governed, completed production run-registry records. The loader verifies each registered file hash and requires a minimal canonical row schema. It never searches for convenient unregistered substitutes. With no completed generation record, it returns no rows and reports **Not generated yet**.

In [ ]:
from run_registry import read_registry
from scripts.validate_sprint1_historical_artifacts import physical_file_sha256

CURRENT_GENERATION_REQUIRED_COLUMNS = {
    "dataset",
    "sample_id",
    "llm_logical_id",
    "context_mode",
    "status",
    "prompt_sha256",
    "decoding_sha256",
    "replica",
    "raw_response",
}
CANONICAL_GENERATION_STATUSES = {"OK", "REFUSAL", "PARSE_FAILURE", "TRUNCATED", "ERROR"}

def latest_pubmedqa_generation_records(registry_path: Path = RUN_REGISTRY_PATH) -> tuple[dict, ...]:
    snapshots = read_registry(registry_path)
    latest_by_run = {}
    for record in snapshots:
        latest_by_run[record["run_id"]] = record
    return tuple(
        record
        for record in latest_by_run.values()
        if record["run_type"] == "GENERATION"
        and record["data"]["dataset"] == "pubmedqa"
        and record["origin"] in {"PROSPECTIVE_BACKFILL", "CURRENT_PROTOCOL"}
    )

def load_registered_generation_rows(records: tuple[dict, ...]) -> pd.DataFrame:
    completed = [record for record in records if record["execution"]["status"] == "COMPLETE"]
    if not completed:
        return pd.DataFrame()
    frames = []
    for record in completed:
        for artifact in record["output"]["artifacts"]:
            relative_path = Path(artifact["path"])
            if relative_path.is_absolute() or ".." in relative_path.parts:
                raise ValueError(f"Unsafe registered output path: {relative_path}")
            path = REPO_ROOT / relative_path
            if not path.is_file():
                raise FileNotFoundError(f"Registered generation artifact is missing: {path}")
            if physical_file_sha256(path) != artifact["sha256"]:
                raise ValueError(f"Registered generation artifact hash mismatch: {path}")
            if path.suffix == ".jsonl":
                frame = pd.read_json(path, lines=True)
            elif path.suffix == ".csv":
                frame = pd.read_csv(path)
            else:
                raise ValueError(f"Unsupported registered generation row format: {path.suffix}")
            missing = sorted(CURRENT_GENERATION_REQUIRED_COLUMNS - set(frame.columns))
            if missing:
                raise ValueError(f"Registered generation rows are missing fields {missing}: {path}")
            if set(frame["status"].dropna()) - CANONICAL_GENERATION_STATUSES:
                raise ValueError(f"Non-canonical generation status in {path}")
            frame = frame.copy()
            frame["registry_run_id"] = record["run_id"]
            frames.append(frame)
    return pd.concat(frames, ignore_index=True) if frames else pd.DataFrame()

current_generation_records = latest_pubmedqa_generation_records()
current_generation_rows = load_registered_generation_rows(current_generation_records)
if current_generation_rows.empty:
    display(Markdown("### Current generation status: Not generated yet"))
else:
    display(Markdown(f"### Current generation status: {len(current_generation_rows):,} registered rows loaded"))

## 11. Planned current analyses

The functions below prepare status accounting, paired `WITH_CONTEXT` minus `WITHOUT_CONTEXT` correctness, retriever and LLM summaries, and the frozen question-level paired percentile bootstrap. They fail on missing columns and return no invented estimates. No current analysis is run until governed results exist.

In [ ]:
def require_columns(frame: pd.DataFrame, columns: set[str], purpose: str) -> None:
    missing = sorted(columns - set(frame.columns))
    if missing:
        raise ValueError(f"{purpose} requires missing columns: {missing}")

def generation_status_counts(frame: pd.DataFrame) -> pd.DataFrame:
    require_columns(frame, {"context_mode", "llm_logical_id", "status"}, "status accounting")
    grouping = ["context_mode", "llm_logical_id"]
    if "retriever" in frame.columns:
        grouping.insert(1, "retriever")
    return frame.groupby(grouping + ["status"], dropna=False).size().rename("N").reset_index()

def paired_with_minus_without(
    frame: pd.DataFrame,
    value_column: str = "pubmedqa_decision_accuracy",
) -> pd.DataFrame:
    required = {"sample_id", "llm_logical_id", "context_mode", "retriever", value_column}
    require_columns(frame, required, "WITH-vs-WITHOUT paired analysis")
    without = frame.loc[frame["context_mode"] == "WITHOUT_CONTEXT", ["sample_id", "llm_logical_id", value_column]].copy()
    if without.duplicated(["sample_id", "llm_logical_id"]).any():
        raise ValueError("WITHOUT_CONTEXT rows are duplicated across the canonical reuse key.")
    without = without.rename(columns={value_column: "without_value"})
    with_context = frame.loc[frame["context_mode"] == "WITH_CONTEXT", ["sample_id", "llm_logical_id", "retriever", value_column]].copy()
    with_context = with_context.rename(columns={value_column: "with_value"})
    paired = with_context.merge(without, on=["sample_id", "llm_logical_id"], how="inner", validate="many_to_one")
    paired = paired.dropna(subset=["with_value", "without_value"])
    paired["paired_difference"] = paired["with_value"] - paired["without_value"]
    return paired

def mean_by_retriever_and_llm(frame: pd.DataFrame, value_column: str) -> pd.DataFrame:
    require_columns(frame, {"context_mode", "retriever", "llm_logical_id", value_column}, "retriever/LLM summary")
    measured = frame.loc[(frame["context_mode"] == "WITH_CONTEXT") & frame[value_column].notna()]
    return measured.groupby(["retriever", "llm_logical_id"])[value_column].agg(["count", "mean"]).reset_index()

def paired_bootstrap_mean_ci(
    paired_differences,
    *,
    resamples: int = 10_000,
    seed: int = 20_260_823,
    confidence: float = 0.95,
) -> dict:
    values = np.asarray(pd.Series(paired_differences).dropna(), dtype=float)
    if values.size == 0 or not np.isfinite(values).all():
        return {"status": "INDETERMINATE", "N": int(values.size), "mean": np.nan, "ci_low": np.nan, "ci_high": np.nan}
    rng = np.random.default_rng(seed)
    bootstrap_means = np.empty(resamples, dtype=float)
    for start in range(0, resamples, 1_000):
        size = min(1_000, resamples - start)
        indices = rng.integers(0, values.size, size=(size, values.size))
        bootstrap_means[start : start + size] = values[indices].mean(axis=1)
    alpha = 1.0 - confidence
    low, high = np.quantile(bootstrap_means, [alpha / 2, 1 - alpha / 2])
    return {"status": "MEASURED", "N": int(values.size), "mean": float(values.mean()), "ci_low": float(low), "ci_high": float(high)}

if current_generation_rows.empty:
    display(Markdown("Planned analysis functions are ready; no current estimates were computed because generation rows do not exist."))

## 12. Sprint 1 status and next step

| Component | Status |
|---|---|
| Historical Sprint 1 | **VERIFIED** |
| Current retrieval artifacts | **LOAD/VALIDATE EXISTING ONLY** |
| Current three-LLM generation | **PENDING** |
| Current evaluation | **PENDING** |
| Sprint 1 freeze | **PENDING** |

The next execution step remains governed completion of missing current Top-20 candidate artifacts outside this notebook, followed by registered current-protocol generation. This notebook must be rerun only to load and analyse those saved artifacts; it must not become an experiment runner.